In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch


plt.rcParams['font.family'] = 'Times New Roman'

# Load table
data = pd.read_csv("merged_tpm_data.tsv", sep="\t")

# PAGM was missing in preliminary figures. It was added manually here
data.loc[data["target_id"] == "mrna-3520", "protein_name"] = "PAGM"

# gene name mapping following Swissprot convention
gene_name_mapping = {
    "mrna-16282": "PIF97", "mrna-13827": "COL12A1-1", "mrna-16835": "COL12A1-2", "mrna-16834": "COL12A1-3",
    "mrna-7898": "CDA1", "mrna-13120": "CDA8", "mrna-1317": "MMP21", "mrna-10071": "MMP3",
    "mrna-14062": "BARHL1", "mrna-3506": "LHX1", "mrna-6488": "HLX1", "mrna-12098": "SHH",
    "mrna-4493": "TMPRSS3", "mrna-4494": "PRSS30", "mrna-6423": "APEH",
    "mrna-8988": "SLC17A5", "mrna-5971": "SLC17A9", "mrna-11829": "AQP9", "mrna-2532": "CPS",
    "mrna-3520": "PAGM"  # New gene
}

# Filter
selected_mrnas = list(gene_name_mapping.keys())
filtered_data = data[data["target_id"].isin(selected_mrnas)].copy()
filtered_data.index = [gene_name_mapping[mrna] for mrna in filtered_data["target_id"]]

# Extract only TPM columns
tpm_columns = [col for col in filtered_data.columns if col.endswith("_tpm")]
expression_matrix = filtered_data[tpm_columns].apply(pd.to_numeric, errors="coerce")

# Normalize rows 
filtered_matrix_normalized = (expression_matrix.sub(expression_matrix.mean(axis=1), axis=0)
                               .div(expression_matrix.std(axis=1), axis=0))

# Rename tissues 
column_name_mapping = {
    "spi_S_tpm": "Siphuncle", "spi_BM_tpm": "Buccal Mass", "spi_RBH_tpm": "Right Brachial Heart",
    "spi_LG_tpm": "Left Gill", "spi_LBH_tpm": "Left Brachial Heart", "spi_RG_tpm": "Right Gill",
    "spi_RWB_tpm": "Right White Body", "spi_AIV_tpm": "Arm IV", "spi_ROL_tpm": "Optic Lobe",
    "sp_ORM_tpm": "Mantle", "sp_TMS_tpm": "Tentacle"
}
filtered_matrix_normalized.rename(columns=column_name_mapping, inplace=True)

# Define gene order
gene_order = [
    "PIF97", "PAGM", "COL12A1-1", "COL12A1-2", "COL12A1-3", "CDA1", "CDA8",
    "MMP21", "MMP3",
    "BARHL1", "LHX1", "HLX1", "SHH",
    "TMPRSS3", "PRSS30", "APEH",
    "SLC17A5", "SLC17A9",
    "AQP9", "CPS"
]
filtered_matrix_normalized = filtered_matrix_normalized.loc[gene_order]

# Classification of genes
classification_labels = [
    "Shell matrix proteins", "Shell matrix proteins", "Shell matrix proteins", "Shell matrix proteins",
    "Shell matrix proteins", "Shell matrix proteins", "Shell matrix proteins",
    "Matrix remodeling enzymes", "Matrix remodeling enzymes",
    "Homeobox genes (TF)", "Homeobox genes (TF)", "Homeobox genes (TF)", "Signaling molecules",
    "Serine proteases", "Serine proteases", "Serine proteases",
    "Ion transport and regulation", "Ion transport and regulation",
    "Buoyancy regulation", "Nitrogen metabolism"
]
classification_colors = {
    "Shell matrix proteins": "#E6AB02",
    "Matrix remodeling enzymes": "#ff7f00",
    "Homeobox genes (TF)": "#4daf4a",
    "Signaling molecules": "#e41a1c",
    "Serine proteases": "#984ea3",
    "Ion transport and regulation": "#a65628",
    "Buoyancy regulation": "#1f78b4",
    "Nitrogen metabolism": "#f781bf"
}
col_colors = pd.Series(
    [classification_colors[label] for label in classification_labels],
    index=gene_order
)

# Transpose and plot heatmap 
transposed_matrix = filtered_matrix_normalized.T

sns.set(font_scale=1.0)
clustermap = sns.clustermap(
    transposed_matrix,
    cmap="YlGnBu",
    linewidths=0.5,
    figsize=(12, 8),
    cbar_kws={'label': 'Scaled Expression', 'orientation': 'vertical'},
    row_cluster=True,
    col_cluster=False,
    xticklabels=True,
    yticklabels=True,
    col_colors=col_colors.values
)

# Formatting axes 
clustermap.ax_col_dendrogram.set_position([0.25, 0.75, 0.65, 0.05])
clustermap.ax_col_colors.set_position([0.25, 0.72, 0.65, 0.02])
clustermap.ax_heatmap.set_position([0.25, 0.15, 0.65, 0.55])

clustermap.ax_heatmap.tick_params(axis='x', labelsize=12, labelrotation=90, pad=3)
clustermap.ax_heatmap.tick_params(axis='y', labelsize=12, labelrotation=0)

for label in clustermap.ax_heatmap.get_xticklabels():
    label.set_fontstyle('italic')
    label.set_horizontalalignment('right')

for label in clustermap.ax_heatmap.get_yticklabels():
    label.set_horizontalalignment('right')

# === Step 9: Colorbar ===
clustermap.cax.set_position([0.08, 0.4, 0.015, 0.2])
clustermap.cax.tick_params(labelsize=14)
clustermap.cax.yaxis.label.set_size(16)

# === Step 10: Legend ===
legend_ax = clustermap.fig.add_axes([1.18, 0.35, 0.1, 0.3])
handles = [
    Patch(facecolor=color, edgecolor="black", label=label)
    for label, color in classification_colors.items()
]
legend_ax.legend(
    handles, classification_colors.keys(),
    title="Classification", loc='center', borderaxespad=0.5, frameon=False,
    title_fontsize=16,
    prop={'family': 'Times New Roman', 'size': 14}
)
legend_ax.axis("off")

# === Step 11: Title and save ===
plt.suptitle("Heatmap of selected genes in the siphuncle of S. spirula",
             x=0.65, y=0.88, fontsize=20, va='center', ha='center')

clustermap.ax_heatmap.tick_params(axis='y', which='both', left=True, right=False)
#plt.savefig("spirula_heatmap.svg", format="svg", bbox_inches='tight') #### Used SVG file in inkscape to manually improved the figure ...
plt.show()
